In [2]:
import string
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

## Download NLTK's resources if they are not present in local machine.

In [3]:
def init_nltk_resources():
    nltk.download('stopwords')
    nltk.download('punkt_tab')
    nltk.download('wordnet')
    nltk.download('omw-1.4')

In [4]:
init_nltk_resources()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Performs

#### 1. punctuation-removal

#### 2. stopwords-removal

#### 3. lemmatization


In [5]:
def preprocess(docs):
    clean_docs = []

    # get a lemmatizer object from NLTK
    lemma = WordNetLemmatizer()

    # get NLTK's list of stopwords
    stop_words = stopwords.words('english')

    # create a mapper that replaces punctuations (defined
    # in string.punctuation) to an empty string
    punc = str.maketrans('', '', string.punctuation)

    for doc in docs:
        # 1. Remove punctuation
        doc_no_punc = doc.translate(punc)
        # convert all characters to lowercase (normalization)
        words = doc_no_punc.lower().split()

        # 2. Remove stop words
        # 3. Lemmatize
        # any word that is not found in NLTK's list of stopwords
        # is lemmatized to its root-form ('v' means 'verb')
        # and stored in the 'words' array
        words = [lemma.lemmatize(word, 'v')
                    for word in words if word not in stop_words]

        # join each word in our list to form back a document
        clean_docs.append(' '.join(words))

    return clean_docs

## Function - BOW
### Performs Bag-Of-Words to extract features from our corpus.

In [6]:
def BOW(docs):
    bow = CountVectorizer()

    # fit → learn vocabulary from docs
    # transform → convert docs into vectors
    # toarray() transforms results in a sparse matrix form to a dense matrix form
    feature_vectors = bow.fit_transform(docs).toarray()

    # returning both feature-vectors and feature-names. the
    # feature-vectors are aligned with the feature-names (vocab)
    return feature_vectors, bow.get_feature_names_out()

## Function - pretty_print
### Display feature-vectors along with feature-names (aka - our vocab)

In [7]:
def pretty_print(feat_vecs, feat_names):
    df = pd.DataFrame(data=feat_vecs,
            index=['doc1', 'doc2', 'doc3'],
            columns=feat_names)

    print(df)

## Preprocess documents

In [8]:
docs = [
    'John has some cats.',
    'Cats, being cats, eat fish.',
    'I ate a big fish.'
]

# data cleansing
clean_docs = preprocess(docs)
print(clean_docs)

['john cat', 'cat cat eat fish', 'eat big fish']


## BOW Example

In [9]:
# getting feature-vectors and feature-names from
# after our BOW process
feat_vecs, feat_names = BOW(clean_docs)

In [ ]:
print(feat_vecs)

In [ ]:
print(feat_names)

In [10]:
# display the count-frequencies against the
# vocabulary (features) and documents
pretty_print(feat_vecs, feat_names)

      big  cat  eat  fish  john
doc1    0    1    0     0     1
doc2    0    2    1     1     0
doc3    1    0    1     1     0


## Function TF-IDF
### Performs TF-IDF to extract features from our corpus.

In [11]:
def TF_IDF(docs):
    tfidf = TfidfVectorizer()

    # toarray() transforms results in a sparse matrix form
    # to a dense matrix form
    # feature_vectors = tfidf.fit(docs).transform(docs).toarray()
    feature_vectors = tfidf.fit_transform(docs).toarray()

    # returning both feature-vectors and feature-names. the
    # feature-vectors are aligned with the feature-names (vocab)
    return feature_vectors, tfidf.get_feature_names_out()

## TF-IDF Example

In [12]:
# getting feature-vectors and feature-names from
# after our TF-IDF process
feat_vecs, feat_names = TF_IDF(clean_docs)

In [ ]:
print(feat_vecs)

In [ ]:
print(feat_names)

In [13]:
pretty_print(feat_vecs, feat_names)

           big       cat       eat      fish      john
doc1  0.000000  0.605349  0.000000  0.000000  0.795961
doc2  0.000000  0.816497  0.408248  0.408248  0.000000
doc3  0.680919  0.000000  0.517856  0.517856  0.000000


## Cosine Similarity Example

In [14]:
clean_docs

['john cat', 'cat cat eat fish', 'eat big fish']

In [15]:
query = ['cats and fish']
clean_query = preprocess(query)
clean_query

['cat fish']

In [16]:
# compute normalized TF-IDF
tfidf = TfidfVectorizer()
tfidf.fit(clean_docs)

fv_corpus = tfidf.transform(clean_docs).toarray()
fv_query = tfidf.transform(clean_query).toarray()

fv = pd.DataFrame(data=fv_query,
                  index=['query string'],
                  columns=tfidf.get_feature_names_out())

print(fv, '\n')

              big       cat  eat      fish  john
query string  0.0  0.707107  0.0  0.707107   0.0 



In [17]:
#compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(fv_query, fv_corpus)

cs = pd.DataFrame(data=similarity,
                  index=['cosine similarity'],
                  columns=['doc1', 'doc2', 'doc3'])

print(cs)

                       doc1      doc2     doc3
cosine similarity  0.428046  0.866025  0.36618
